In [ ]:
# Mock Data Generator (Smart Conveyor Monitoring System)
Simulates FB1 tag shape (Motor_Run, SystemReady, FaultActive, ItemCount,
TempAlarmHigh, TempAlarmLow) using a random-walk temperature model.
OPC-UA live feed is architecturally blocked (see project README) —
this generates realistic mock data so CSV logging / alarm detection /
dashboard can be built and tested now.
## CSV Logging
Appends readings to a persistent log (real historian behavior — survives
kernel restarts, never auto-wiped). Uses csv.DictWriter for safe column
alignment.
## """Full pipeline demo: generate -> print -> log -> sleep -> repeat."""

In [10]:
import time
import random
import csv
import os
from datetime import datetime

In [11]:
def initial_state():
    """Starting values for one mock conveyor. Call once per machine."""
    return{
    "temp": 22.0,
    "item_count": 0,
    "fault_active": False,
    }

In [12]:
def next_reading(state, fault_chance=0.02, reset_fault=False):
    """
    Advance one mock conveyor by a single time step.
    Returns a NEW state dict — does not mutate the input.
    """
    fault_active = state["fault_active"]
    if reset_fault:
        fault_active = False

    step = random.uniform(-0.5, 0.5)
    new_temp = state["temp"] + step
    new_temp = max(0.0, min(95.0, new_temp)) # keep in physically plausible range
    new_temp = round(new_temp, 2) #realistic sensor precision, not 15 decimals

    temp_alarm_high = new_temp > 80.0
    temp_alarm_low = new_temp < 5.0

    if not fault_active and random.random() < fault_chance:
        fault_active = True

    motor_run = not fault_active
    item_count = state["item_count"]
    if motor_run:
        item_count += 1
        if item_count > 9999:
            item_count = 0


    system_ready = (not fault_active) and (not temp_alarm_high) and (not temp_alarm_low)

    return{
    "temp" : new_temp,
    "item_count" : item_count,
    "fault_active" : fault_active,
    "motor_run" : motor_run,
    "system_ready" : system_ready,
    "temp_alarm_high" : temp_alarm_high,
    "temp_alarm_low" : temp_alarm_low,
    }

In [4]:
LOG_FILE = "pipeline_test.csv"
FIELDNAMES = ["timestamp", "temp", "item_count", "motor_run", "system_ready", "fault_active", "temp_alarm_high", "temp_alarm_low"]

def log_reading(state, log_file=LOG_FILE):
    """Append one conveyor reading to the persistent CSV log.
    Writes the header row only if the file doesn't exist yet.
    """
    file_exists = os.path.exists(log_file)
    with open(log_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()
        row = {"timestamp": datetime.now().isoformat(timespec="seconds")}
        row.update(state)
        writer.writerow(row)

In [16]:
def run_pipeline(num_readings=5, poll_interval=1, fault_chance=0.2):
    """Run the mock pipeline: generate, display, log, wait, repeat."""
    state = initial_state()
    for i in range(num_readings):
        state = next_reading(state, fault_chance=fault_chance)
        status = "FAULT!" if state ["fault_active"] else "ok"
        print(f"[{i}] temp={state['temp']:6.2f} count={state['item_count']:d} status={status}")
        log_reading(state)
        if i < num_readings -1:
            time.sleep(poll_interval)
    print("Pipeline run complete. Logged to", LOG_FILE)

if __name__=="__main__":
    if os.path.exists(LOG_FILE):
        os.remove(LOG_FILE) #clean slate for this demo only
    run_pipeline(num_readings=20, poll_interval=2)

[0] temp= 22.38 count=1 status=ok
[1] temp= 22.62 count=2 status=ok
[2] temp= 22.26 count=3 status=ok
[3] temp= 21.86 count=4 status=ok
[4] temp= 22.19 count=5 status=ok
[5] temp= 22.17 count=5 status=FAULT!
[6] temp= 22.60 count=5 status=FAULT!
[7] temp= 22.47 count=5 status=FAULT!
[8] temp= 22.10 count=5 status=FAULT!
[9] temp= 22.37 count=5 status=FAULT!
[10] temp= 22.79 count=5 status=FAULT!
[11] temp= 22.94 count=5 status=FAULT!
[12] temp= 22.69 count=5 status=FAULT!
[13] temp= 22.30 count=5 status=FAULT!
[14] temp= 22.20 count=5 status=FAULT!
[15] temp= 22.31 count=5 status=FAULT!
[16] temp= 22.03 count=5 status=FAULT!
[17] temp= 22.22 count=5 status=FAULT!
[18] temp= 22.46 count=5 status=FAULT!
[19] temp= 22.89 count=5 status=FAULT!
Pipeline run complete. Logged to pipeline_test.csv


In [ ]:
## Summary — Mock Data Generator
- Built `next_reading()`: random-walk temp (not fresh-random) so alarm
  detection can be tested against continuous, physically plausible data.
- State passed as dict in/out (not global) — enables multi-machine sim later.
- Fault latches until explicit `reset_fault=True` — matches FB1's
  reset-dominant E-Stop behavior.
- Temp clamped [0, 95] — prevents unbounded random-walk drift outside
  any real conveyor's operating range.
- Rounded to round(temp, 2) at the source — realistic sensor precision.

In [ ]:
## Summary — CSV Logging
- Diagnosed "w" mode inside a loop truncating the file each write — only
  last row survived. Fixed by opening once (or using "a" append mode).
- Chose append mode ("a") over fresh-write ("w") to match real historian
  behavior: data persists across kernel restarts, never silently wiped.
- csv.DictWriter matches columns by field name, not position — avoids
  silent column-misalignment bugs (caught a fieldname typo at write-time
  instead of silently writing wrong columns).

In [ ]:
## Summary — Polling Loop (time.sleep)
- Wrapped generator + logger in run_pipeline(num_readings, poll_interval,
  fault_chance) — reusable, adjustable via parameters.
- time.sleep() placed INSIDE the for loop (after print+log, before next
  iteration) — placing it outside the loop only runs it once total,
  not once per reading. Confirmed via broken-vs-correct comparison.
- Skip sleep after the final reading (if i < num_readings - 1) — avoids
  a pointless wait with nothing left to do.
- This loop shape (read -> report -> log -> wait -> repeat) matches a
  real OPC-UA polling client — next_reading() is the only piece that
  changes when live PLCSIM connectivity is eventually available.

In [ ]:
## Concept: if __name__ == "__main__"
- __name__ is a variable Python sets automatically per file.
- Running a file directly -> __name__ == "__main__" -> block runs.
- Importing the file from elsewhere -> __name__ == filename -> block
  is skipped, but function/class definitions above it still load.
- Prevents test/demo code from firing automatically when this file's
  functions get reused (e.g. imported into a future dashboard script).